In [89]:
!pip install gradio transformers torch sentence-transformers nltk spacy scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


**Tab 1: Text Preprocessing (Tokenization)**

Imports et téléchargement NLTK


In [90]:
import gradio as gr
import nltk
from transformers import AutoTokenizer

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Chargement des modèles HuggingFace


In [91]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

Fonction de tokenisation

In [92]:
def tokenize_text(text):
    if not text.strip():
        return "", "", "", "", "", "", "", ""

    nltk_tokens = nltk.word_tokenize(text)
    nltk_display = " | ".join(nltk_tokens)
    nltk_count = f"Total : {len(nltk_tokens)} tokens"

    bert_tokens = bert_tokenizer.tokenize(text)
    bert_display = " | ".join(bert_tokens)
    bert_count = f"Total : {len(bert_tokens)} tokens"

    t5_tokens = t5_tokenizer.tokenize(text)
    t5_display = " | ".join(t5_tokens)
    t5_count = f"Total : {len(t5_tokens)} tokens"

    gpt2_tokens = gpt2_tokenizer.tokenize(text)
    gpt2_display = " | ".join(gpt2_tokens)
    gpt2_count = f"Total : {len(gpt2_tokens)} tokens"

    return nltk_count, nltk_display, bert_count, bert_display, t5_count, t5_display, gpt2_count, gpt2_display

**Tab 2: Sentiment & Emotion Analysis**

Imports


In [93]:

from transformers import pipeline

In [94]:

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
sentiment_roberta = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fonction d'analyse

In [95]:
def analyze_sentiment(text):
    # Analyse avec distilbert
    result = sentiment_model(text)[0]
    label = result['label']
    score = result['score']
    if label == "POSITIVE":
        pos = round(score * 100)
        neg = round((1 - score) * 100)
    else:
        neg = round(score * 100)
        pos = round((1 - score) * 100)

    # Deuxieme modele roberta
    r2_all = sentiment_roberta(text, top_k=None)
    rob_pos = rob_neu = rob_neg = "N/A"
    for item in r2_all:
        lbl = item['label'].lower()
        pct = round(item['score'] * 100)
        if lbl == "positive":
            rob_pos = f" Positif : {pct}%"
        elif lbl == "neutral":
            rob_neu = f" Neutre : {pct}%"
        elif lbl == "negative":
            rob_neg = f" Négatif : {pct}%"

    return f" Positif : {pos}%", f" Négatif : {neg}%", rob_pos, rob_neu, rob_neg

**Tab 3: Semantic Similarity**

Imports

In [96]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

 Chargement du modèle

In [97]:
similarity_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
similarity_model_2 = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L3-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L3-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fonction de similarité


In [98]:
def compute_similarity(text1, text2):
    if not text1.strip() or not text2.strip():
        return " Veuillez entrer deux textes.", "", " Veuillez entrer deux textes.", ""

    # Modèle 1
    embedding1 = similarity_model.encode([text1])
    embedding2 = similarity_model.encode([text2])
    score1 = round(float(cosine_similarity(embedding1, embedding2)[0][0]) * 100, 2)

    if score1 >= 80:
        interpretation1 = " Très similaires — Les deux phrases ont presque le même sens."
    elif score1 >= 50:
        interpretation1 = " Moyennement similaires — Les phrases partagent des idées communes."
    elif score1 >= 20:
        interpretation1 = " Peu similaires — Les phrases ont des sens assez différents."
    else:
        interpretation1 = " Très différentes — Les phrases n'ont presque rien en commun."

    # Modèle 2
    embedding3 = similarity_model_2.encode([text1])
    embedding4 = similarity_model_2.encode([text2])
    score2 = round(float(cosine_similarity(embedding3, embedding4)[0][0]) * 100, 2)

    if score2 >= 80:
        interpretation2 = " Très similaires — Les deux phrases ont presque le même sens."
    elif score2 >= 50:
        interpretation2 = " Moyennement similaires — Les phrases partagent des idées communes."
    elif score2 >= 20:
        interpretation2 = " Peu similaires — Les phrases ont des sens assez différents."
    else:
        interpretation2 = " Très différentes — Les phrases n'ont presque rien en commun."

    return f" Score : {score1}%", interpretation1, f" Score : {score2}%", interpretation2

**Tab 4 : Zero-Shot Topic Classification**

Imports

In [99]:
from transformers import pipeline

Chargement du modèle

In [100]:
classifier_model = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)
classifier_bart = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Fonction de classification

In [101]:
def classify_text(text, labels_str):
    if not text.strip() or not labels_str.strip():
        return "Veuillez entrer un texte et des categories.", "Veuillez entrer un texte et des categories."

    labels = [label.strip() for label in labels_str.split(",")]

    # classification avec xlm
    result1 = classifier_model(text, candidate_labels=labels, multi_label=False)
    output1 = f"{result1['labels'][0]} : {round(result1['scores'][0] * 100, 2)}%"

    # classification avec bart
    result2 = classifier_bart(text, candidate_labels=labels, multi_label=False)
    output2 = f"{result2['labels'][0]} : {round(result2['scores'][0] * 100, 2)}%"

    return output1, output2

**Tab 5: Innovation **


Imports

In [102]:
from transformers import pipeline

Chargement du modèle

In [103]:
ner_model = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fonction NER

In [104]:
def extract_entities(text):
    if not text.strip():
        return " Veuillez entrer un texte."

    entities = ner_model(text)

    if not entities:
        return "Aucune entité détectée dans ce texte."


    icons = {
        "PER": " PERSONNE",
        "ORG": " ORGANISATION",
        "LOC": " LIEU",
        "MISC": " DIVERS"
    }


    grouped = {}
    for ent in entities:
        entity_type = ent["entity_group"]
        if entity_type not in grouped:
            grouped[entity_type] = []
        grouped[entity_type].append(
            f"{ent['word']} ({round(ent['score']*100, 2)}%)"
        )

    output = ""
    for entity_type, words in grouped.items():
        label = icons.get(entity_type, entity_type)
        output += f"{label} :\n"
        for word in words:
            output += f"   → {word}\n"
        output += "\n"

    output += f"{'─'*40}\n"
    output += f" Total : {len(entities)} entités détectées"

    return output

**Question Answering**

Chargement du modèle

In [105]:
qa_model = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fonction

In [106]:
def answer_question(context, question):
    if not context.strip() or not question.strip():
        return "Veuillez entrer un contexte et une question."

    result = qa_model(question=question, context=context)
    answer = result["answer"]
    score = round(result["score"] * 100, 2)

    return f" Réponse : {answer}\n Confiance : {score}%"

**Language Detection**

 Modèle

In [107]:
lang_model = pipeline(
    "text-classification",
    model="papluca/xlm-roberta-base-language-detection"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: papluca/xlm-roberta-base-language-detection
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fonction

In [108]:
def detect_language(text):
    if not text.strip():
        return "Veuillez entrer un texte."

    result = lang_model(text)[0]
    language = result["label"]
    score = round(result["score"] * 100, 2)

    return f" Langue détectée : {language}\n Confiance : {score}%"

Interface

In [109]:
with gr.Blocks() as demo1:
    gr.Markdown("---")
    gr.Markdown("###  Entrez votre texte")
    text_input = gr.Textbox(
        value="Biotechnology is revolutionizing the way we live",
        label="Entrez votre texte",
        lines=2,
        placeholder="Tapez votre texte ici..."
    )
    btn = gr.Button(" Tokeniser", variant="primary")
    gr.Markdown("---")
    gr.Markdown("###  Methode 1 - NLTK (Word-level)")
    with gr.Row():
        nltk_count = gr.Textbox(label="Nombre de tokens", scale=1)
        nltk_out = gr.Textbox(label="Tokens", scale=3)
    gr.Markdown("###  Methode 2 - BERT (Subword/WordPiece)")
    with gr.Row():
        bert_count = gr.Textbox(label="Nombre de tokens", scale=1)
        bert_out = gr.Textbox(label="Tokens", scale=3)
    gr.Markdown("###  Methode 3 - T5 (SentencePiece)")
    with gr.Row():
        t5_count = gr.Textbox(label="Nombre de tokens", scale=1)
        t5_out = gr.Textbox(label="Tokens", scale=3)
    gr.Markdown("###  Methode 4 - GPT-2 (BPE)")
    with gr.Row():
        gpt2_count = gr.Textbox(label="Nombre de tokens", scale=1)
        gpt2_out = gr.Textbox(label="Tokens", scale=3)
    btn.click(
        tokenize_text,
        inputs=text_input,
        outputs=[nltk_count, nltk_out, bert_count, bert_out, t5_count, t5_out, gpt2_count, gpt2_out]
    )

with gr.Blocks() as demo2:
    gr.Markdown("---")
    gr.Markdown("###  Texte a analyser")
    text_input = gr.Textbox(
        value="The course was pretty good, I learned some interesting things",
        label="Entrez votre texte",
        lines=2,
        placeholder="Tapez votre texte ici..."
    )
    btn = gr.Button(" Analyser le sentiment", variant="primary")
    gr.Markdown("---")
    gr.Markdown("###  Modele 1 - DistilBERT (SST-2)")
    with gr.Row():
        pos_out = gr.Textbox(label="Score Positif")
        neg_out = gr.Textbox(label="Score Negatif")
    gr.Markdown("###  Modele 2 - RoBERTa (Twitter)")
    with gr.Row():
        rob_pos = gr.Textbox(label="Score Positif")
        rob_neu = gr.Textbox(label="Score Neutre")
        rob_neg = gr.Textbox(label="Score Negatif")
    btn.click(
        analyze_sentiment,
        inputs=text_input,
        outputs=[pos_out, neg_out, rob_pos, rob_neu, rob_neg]
    )

with gr.Blocks() as demo3:
    gr.Markdown("> Mesure la distance semantique entre deux textes.")
    gr.Markdown("---")
    gr.Markdown("###  Entrez vos deux textes")
    with gr.Row():
        text1 = gr.Textbox(
            value="Education is important for every young person",
            label="Texte 1",
            lines=3,
            placeholder="Entrez le premier texte..."
        )
        text2 = gr.Textbox(
            value="Students went back to school after the holidays",
            label="Texte 2",
            lines=3,
            placeholder="Entrez le deuxieme texte..."
        )
    btn = gr.Button(" Calculer la similarite", variant="primary")
    gr.Markdown("---")
    gr.Markdown("###  Modele 1 - all-MiniLM-L6-v2")
    with gr.Row():
        score_out = gr.Textbox(label="Score de similarite")
        interpretation_out = gr.Textbox(label="Interpretation")
    gr.Markdown("###  Modele 2 - paraphrase-MiniLM-L3-v2")
    with gr.Row():
        score_out2 = gr.Textbox(label="Score de similarite")
        interpretation_out2 = gr.Textbox(label="Interpretation")
    btn.click(
        compute_similarity,
        inputs=[text1, text2],
        outputs=[score_out, interpretation_out, score_out2, interpretation_out2]
    )

with gr.Blocks() as demo4:
    gr.Markdown("---")
    gr.Markdown("###  Texte a classifier")
    text_input = gr.Textbox(
        value="Artificial intelligence is revolutionizing the way we use computers and smartphones",
        label="Entrez votre texte",
        lines=3,
        placeholder="Tapez votre texte ici..."
    )
    gr.Markdown("###  Categories")
    labels_input = gr.Textbox(
        value="Politics, Technology, Sports, Environment, Health",
        label="Entrez vos categories (separees par des virgules)",
        placeholder="Ex: Politics, Sports, Technology..."
    )
    btn = gr.Button(" Classifier", variant="primary")
    gr.Markdown("---")
    gr.Markdown("###  Modele 1 - XLM-RoBERTa XNLI")
    output1 = gr.Textbox(label="Scores de confiance", lines=8)
    gr.Markdown("###  Modele 2 - BART-large-MNLI")
    output2 = gr.Textbox(label="Scores de confiance", lines=8)
    btn.click(
        classify_text,
        inputs=[text_input, labels_input],
        outputs=[output1, output2]
    )

with gr.Blocks() as demo5:
    gr.Markdown("###  Tab 5 - Named Entity Recognition (NER)")
    gr.Markdown("---")
    gr.Markdown("###  Texte a analyser")
    text_input = gr.Textbox(
        value="Elon Musk founded SpaceX in 2002 in California.",
        label="Entrez votre texte",
        lines=3,
        placeholder="Tapez votre texte ici..."
    )
    btn = gr.Button(" Extraire les entites", variant="primary")
    gr.Markdown("---")
    gr.Markdown("###  Entites detectees")
    output = gr.Textbox(label="Resultats", lines=12)
    gr.Markdown("---")
    gr.Markdown("""
    ###  Types d'entites detectees
    -  **PER** -> Personnes
    -  **ORG** -> Organisations
    -  **LOC** -> Lieux
    -  **MISC** -> Divers
    """)
    btn.click(
        extract_entities,
        inputs=text_input,
        outputs=output
    )

with gr.Blocks() as demo6:
    gr.Markdown("> Repond a une question basee sur un contexte fourni.")
    gr.Markdown("---")
    gr.Markdown("###  Contexte")
    context_input = gr.Textbox(
        value="Elon Musk founded SpaceX in 2002 in California. He also founded Tesla in 2003.",
        label="Entrez votre contexte",
        lines=4,
        placeholder="Entrez le texte de contexte ici..."
    )
    gr.Markdown("###  Question")
    question_input = gr.Textbox(
        value="When was SpaceX founded?",
        label="Entrez votre question",
        placeholder="Posez votre question ici..."
    )
    btn = gr.Button(" Repondre", variant="primary")
    gr.Markdown("---")
    gr.Markdown("###  Reponse")
    output = gr.Textbox(label="Resultat", lines=3)
    btn.click(
        answer_question,
        inputs=[context_input, question_input],
        outputs=output
    )

with gr.Blocks() as demo7:
    gr.Markdown("> Detecte automatiquement la langue d'un texte.")
    gr.Markdown("---")
    gr.Markdown("###  Entrez votre texte")
    text_input = gr.Textbox(
        value="Bonjour, comment allez-vous aujourd'hui?",
        label="Entrez votre texte",
        lines=3,
        placeholder="Tapez votre texte ici..."
    )
    btn = gr.Button(" Detecter la langue", variant="primary")
    gr.Markdown("---")
    gr.Markdown("###  Resultat")
    output = gr.Textbox(label="Langue detectee", lines=3)
    btn.click(
        detect_language,
        inputs=text_input,
        outputs=output
    )

dark_theme = gr.themes.Base(
    primary_hue="pink",
    neutral_hue="slate"
).set(
    body_background_fill="#000000",
    body_background_fill_dark="#000000",
    block_background_fill="#111111",
    block_background_fill_dark="#111111",
    body_text_color="#ffffff",
    body_text_color_dark="#ffffff",
    block_label_text_color="#ffffff",
    block_label_text_color_dark="#ffffff",
    input_background_fill="#1a1a1a",
    input_background_fill_dark="#1a1a1a",
)



In [110]:
app = gr.TabbedInterface(
    [demo1, demo2, demo3, demo4, demo5, demo6, demo7],
    ["Tokenization", " Sentiment", " Similarity", " Zero-Shot", " NER", " Q&A", " Lang Detection"],
    theme=dark_theme
)
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f994310ca9e5fc136.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
